# Predicción de graduación universitaria mediante regresión logística


El ejercicio utiliza un conjunto de datos **sintéticos** de 2.000 estudiantes. El objetivo consiste en estimar si un estudiante se gradúa (`1`) o no se gradúa dentro del período observado (`0`).

## 1. Bibliotecas

Se utilizan `pandas` y `numpy` para el tratamiento de los datos, `matplotlib` para las gráficas y `scikit-learn` para el preprocesamiento, el entrenamiento y la evaluación.

En un entorno nuevo, se instalan con:

```bash
pip install pandas numpy matplotlib scikit-learn jupyter
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, log_loss, matthews_corrcoef,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 20)

## 2. Lectura y exploración inicial

El archivo CSV debe encontrarse en la misma carpeta que el notebook. Se revisan su estructura, los valores ausentes y la distribución de la variable que se desea predecir.

In [ ]:
ruta = Path("estudiantes_graduacion.csv")
df = pd.read_csv(ruta)
print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas")
display(df.head())
print("\nValores ausentes por columna:")
print(df.isna().sum().to_string())

In [ ]:
distribucion = df["se_graduo"].value_counts().sort_index()
print(pd.DataFrame({
    "Cantidad": distribucion,
    "Porcentaje": (distribucion / len(df) * 100).round(2)
}).rename(index={0: "No se graduó", 1: "Se graduó"}))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["No se graduó (0)", "Se graduó (1)"], distribucion.values)
ax.set_ylabel("Número de estudiantes")
ax.set_title("Distribución de la variable objetivo")
plt.tight_layout()
plt.show()

**Interpretación:** se identifica qué clase es más frecuente. Esta proporción influye en la lectura del *accuracy*: un modelo que siempre predice la clase mayoritaria puede alcanzar cierta exactitud sin aprender a reconocer bien la clase minoritaria.

## 3. Definición de variables y partición

Se elimina `id_estudiante` porque identifica registros, pero no constituye una característica predictiva académica. La variable `genero` se codifica mediante *one-hot encoding*; las variables numéricas se estandarizan. El preprocesamiento se ajusta **únicamente con los datos de entrenamiento** mediante un `Pipeline`, evitando la fuga de información desde el conjunto de prueba.

La división estratificada conserva aproximadamente la proporción de graduados y no graduados en ambos subconjuntos.

In [ ]:
X = df.drop(columns=["id_estudiante", "se_graduo"])
y = df["se_graduo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

In [ ]:
columnas_categoricas = ["genero"]
columnas_numericas = X.select_dtypes(include="number").columns.tolist()

preprocesador = ColumnTransformer(transformers=[
    ("numericas", Pipeline(steps=[
        ("imputacion", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]), columnas_numericas),
    ("categoricas", Pipeline(steps=[
        ("imputacion", SimpleImputer(strategy="most_frequent")),
        ("codificacion", OneHotEncoder(handle_unknown="ignore")),
    ]), columnas_categoricas),
])

modelo = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42)),
])

print(f"Entrenamiento: {len(X_train)} estudiantes")
print(f"Prueba: {len(X_test)} estudiantes")
print(f"Proporción clase 1 en entrenamiento: {y_train.mean():.2%}")
print(f"Proporción clase 1 en prueba: {y_test.mean():.2%}")

## 4. Entrenamiento y obtención de predicciones

La regresión logística estima una **probabilidad** entre 0 y 1 para cada registro. Con el umbral predeterminado de 0,50, una probabilidad igual o superior a ese valor se convierte en la clase `1`; de lo contrario, se asigna la clase `0`.

El umbral de 0,50 se utiliza como punto de partida didáctico y no debe considerarse automáticamente óptimo.

In [ ]:
modelo.fit(X_train, y_train)
probabilidades = modelo.predict_proba(X_test)[:, 1]
umbral = 0.50
predicciones = (probabilidades >= umbral).astype(int)

comparacion = pd.DataFrame({
    "Real": y_test.to_numpy(),
    "Probabilidad_graduacion": probabilidades.round(3),
    "Prediccion": predicciones,
})
display(comparacion.head(10))

## 5. Matriz de confusión: punto de partida de la evaluación

La **clase positiva** se define como `1 = se graduó`. Por tanto:

| Concepto | Significado en el ejercicio |
|---|---|
| **VP (verdadero positivo)** | Se predijo graduación y el estudiante efectivamente se graduó. |
| **VN (verdadero negativo)** | Se predijo no graduación y el estudiante efectivamente no se graduó. |
| **FP (falso positivo)** | Se predijo graduación, pero el estudiante no se graduó. |
| **FN (falso negativo)** | Se predijo no graduación, pero el estudiante sí se graduó. |

La matriz permite observar **qué errores comete** el modelo, información que no puede deducirse a partir de una sola métrica global.

In [ ]:
matriz = confusion_matrix(y_test, predicciones, labels=[0, 1])
VN, FP, FN, VP = matriz.ravel()
print(f"Verdaderos negativos (VN): {VN}")
print(f"Falsos positivos (FP):    {FP}")
print(f"Falsos negativos (FN):    {FN}")
print(f"Verdaderos positivos (VP): {VP}")

ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["No se graduó (0)", "Se graduó (1)"],
).plot(values_format="d", cmap="Blues")
plt.title("Matriz de confusión — conjunto de prueba")
plt.tight_layout()
plt.show()

## 6. Accuracy: proporción total de aciertos

Indica qué proporción de los estudiantes del conjunto de prueba fue clasificada correctamente, considerando ambas clases. Un `accuracy` de 0,85 significaría 85 predicciones correctas por cada 100 estudiantes evaluados, aproximadamente.

In [ ]:
accuracy = accuracy_score(y_test, predicciones)
accuracy_base = max(y_test.mean(), 1 - y_test.mean())
print(f"Accuracy del modelo: {accuracy:.2%}")
print(f"Accuracy de predecir siempre la clase mayoritaria: {accuracy_base:.2%}")
print(f"Interpretación: el modelo clasifica correctamente {accuracy:.1%} de los casos de prueba.")

## 7. Precision: fiabilidad de las predicciones positivas

De todos los estudiantes que el modelo **predice que se graduarán**, qué proporción realmente se gradúa. Por ejemplo, `precision = 0,80` significa que 80 de cada 100 predicciones de graduación resultan correctas.

Es importante cuando interesa reducir las **falsas predicciones de graduación** (FP). La precisión no indica cuántos graduados reales logra encontrar el modelo.

In [ ]:
precision = precision_score(y_test, predicciones, zero_division=0)
print(f"Precision: {precision:.2%}")
print(f"Interpretación: de los {VP+FP} estudiantes pronosticados como graduados, "
      f"{VP} efectivamente se graduaron ({precision:.1%}).")

## 8. Recall o sensibilidad: cobertura de los graduados reales

De todos los estudiantes que **realmente se graduaron**, qué proporción fue identificada correctamente. Si `recall = 0,90`, se detectan 90 de cada 100 graduados reales.

Importante cuando se desea evitar que un graduado real sea clasificado como no graduado (FN). No indica por sí solo si las predicciones positivas son fiables.

In [ ]:
recall = recall_score(y_test, predicciones, zero_division=0)
print(f"Recall o sensibilidad: {recall:.2%}")
print(f"Interpretación: de los {VP+FN} graduados reales, "
      f"el modelo identificó {VP} ({recall:.1%}).")

## 9. Especificidad: reconocimiento de los no graduados


De los estudiantes que **no se graduaron**, qué proporción fue clasificada correctamente como no graduada. Una especificidad de 0,75 indica que se reconoce correctamente al 75 % de los casos negativos reales.

Relevante cuando se interesa reconocer correctamente a quienes no se gradúan. Dado que la clase positiva definida es «se graduó», la especificidad analiza el rendimiento sobre la clase `0`.

In [ ]:
especificidad = VN / (VN + FP) if (VN + FP) else 0.0
print(f"Especificidad: {especificidad:.2%}")
print(f"Interpretación: de los {VN+FP} no graduados reales, "
      f"el modelo identificó {VN} ({especificidad:.1%}).")

## 10. F1-score: equilibrio entre precision y recall

Representa la media armónica de `precision` y `recall`. Una cifra alta requiere que ambas medidas sean razonablemente altas. Si una es baja, el F1 disminuye.

F1 no incluye directamente los verdaderos negativos, por lo que conviene interpretarlo junto con la especificidad y la matriz de confusión.

In [ ]:
f1 = f1_score(y_test, predicciones, zero_division=0)
print(f"F1-score: {f1:.3f}")
print(f"Interpretación: el equilibrio entre precision ({precision:.3f}) "
      f"y recall ({recall:.3f}) produce un F1 de {f1:.3f}.")

## 11. Balanced accuracy: rendimiento equilibrado entre clases

Promedia el acierto dentro de las dos clases. Es especialmente útil si existe desbalance de graduados y no graduados.

Un valor de 0,50 puede corresponder a un clasificador trivial que siempre predice una sola clase en un problema binario con ambas clases presentes. No garantiza por sí solo un buen modelo.

In [ ]:
balanced_accuracy = balanced_accuracy_score(y_test, predicciones)
print(f"Balanced accuracy: {balanced_accuracy:.2%}")
print(f"Interpretación: el promedio del recall de graduados y no graduados es "
      f"{balanced_accuracy:.1%}.")

## 12. ROC-AUC: capacidad de distinguir graduados de no graduados

La curva **ROC** representa la sensibilidad frente a la tasa de falsos positivos (`1 - especificidad`) cuando varía el umbral de decisión.

El **ROC-AUC** mide la capacidad del modelo para ordenar por encima a un graduado real frente a un no graduado real elegido al azar. Un valor próximo a 1 indica buena discriminación; 0,50 corresponde al rendimiento de una ordenación aleatoria en términos de AUC.

**Precaución:** el ROC-AUC no mide directamente la calibración de las probabilidades y puede parecer favorable incluso cuando el desempeño sobre una clase minoritaria requiere atención.

In [ ]:
auc_roc = roc_auc_score(y_test, probabilidades)
fpr, tpr, _ = roc_curve(y_test, probabilidades)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"Regresión logística (AUC = {auc_roc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Referencia aleatoria")
ax.set(xlabel="Tasa de falsos positivos (1 - especificidad)",
       ylabel="Sensibilidad (recall)", title="Curva ROC")
ax.legend()
plt.tight_layout()
plt.show()
print(f"ROC-AUC: {auc_roc:.3f}")
print("Interpretación: el valor resume la discriminación del modelo al variar el umbral.")

## 13. Resumen de métricas

Se reúnen las medidas para facilitar su comparación. **No todas utilizan la misma escala ni tienen la misma dirección de mejora**: la pérdida logarítmica se busca minimizar, y el resto de las métricas presentadas se procura maximizar.

In [ ]:
resumen = pd.DataFrame([
    ("Accuracy", accuracy, "Mayor es mejor"),
    ("Precision (clase 1)", precision, "Mayor es mejor"),
    ("Recall / Sensibilidad (clase 1)", recall, "Mayor es mejor"),
    ("Especificidad (clase 0)", especificidad, "Mayor es mejor"),
    ("F1-score (clase 1)", f1, "Mayor es mejor"),
    ("Balanced accuracy", balanced_accuracy, "Mayor es mejor"),
    ("ROC-AUC", auc_roc, "Mayor es mejor"),
    ("Average Precision", ap, "Mayor es mejor"),
    ("Log loss", perdida_log, "Menor es mejor"),
    ("MCC", mcc, "Mayor es mejor"),
], columns=["Métrica", "Resultado", "Criterio"])
display(resumen.style.format({"Resultado": "{:.4f}"}))

print("Reporte por clase (0 = no graduación, 1 = graduación):")
print(classification_report(
    y_test, predicciones,
    labels=[0, 1], target_names=["No se graduó", "Se graduó"],
    zero_division=0
))

## 14. Ejemplo de predicción para un estudiante nuevo

Se construye un registro ficticio con exactamente las mismas variables utilizadas durante el entrenamiento. El modelo devuelve tanto la probabilidad estimada como la clase asignada con el umbral de 0,50.

**Advertencia:** la probabilidad constituye una salida estadística del modelo sintético; no debe utilizarse para tomar decisiones sobre estudiantes reales.

In [ ]:
nuevo_estudiante = pd.DataFrame([{
    "edad_ingreso": 20,
    "genero": "F",
    "promedio_secundaria": 78,
    "ingreso_familiar": 5200,
    "trabaja": 0,
    "horas_trabajo_semana": 0,
    "beca": 1,
    "asistencia_porcentaje": 90,
    "promedio_primer_semestre": 82,
    "materias_inscritas": 6,
    "materias_aprobadas": 5,
    "deuda_universidad": 0,
}])

probabilidad_nueva = modelo.predict_proba(nuevo_estudiante)[0, 1]
clase_nueva = int(probabilidad_nueva >= umbral)
print(f"Probabilidad estimada de graduación: {probabilidad_nueva:.2%}")
print(f"Clase estimada con umbral {umbral:.2f}: {clase_nueva}")